# 강의 03 · 실습 4 — RAG 에이전트 서비스 · (5) 고난도 II

## 1. 문제상황

- 구름월드 홈페이지 팀은 안내 서비스의 답 옆에 「근거: FAQ 8번(티켓)」처럼 출처를 표시하려 합니다.
- 지금 서비스 응답은 답 문장 하나뿐이라 어느 FAQ 항목을 근거로 했는지 화면에 보여 줄 수 없습니다.
- 검색 도구는 청크(chunk)를 문자열로 이어 모델에게 넘기므로, 어느 청크가 쓰였는지가 루프 밖으로 나오지 않습니다.
- 담당자는 검색에서 통과한 청크의 행 번호와 카테고리를 모아 응답에 출처 목록으로 함께 돌려주기를 원합니다. 근거가 없으면 출처 목록은 비어 있어야 합니다.

## 2. 문제와 목표

- **문제**: 응답에 출처가 없고, 도구가 어느 청크를 근거로 넘겼는지 루프 밖에서 알 수 없습니다.
- **목표**: 검색 도구가 통과시킨 청크의 메타데이터(행 번호·카테고리)를 루프가 모아 두고, 응답 모양을 답과 출처 목록 두 칸으로 바꿔 답과 출처 목록을 함께 돌려주는 서비스를 만듭니다.
    - 메타데이터: 청크의 행 번호와 카테고리를 「행 N · 카테고리」 문자열로.
    - 응답 두 칸: 답(문자열)과 출처 목록(문자열 리스트).
    - 주어진 것: 데이터 파일 `day05_faq_구름월드.csv`(32행, 행마다 `[카테고리] Q: … A: …` 본문과 행 번호·카테고리 메타데이터), 임베딩 모델 `text-embedding-3-small`, 저장소 디렉터리 `chroma_db`.
    - 시스템 프롬프트와 고정 안내 문장은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**: 환불 질문의 응답에 답과 함께 출처 목록이 있고 출처는 「행 번호 · 카테고리」 형식(예: 행 5 · 티켓)으로 1개 이상이며, 인사말과 문서 밖 질문의 출처 목록은 비어 있고, 문서 밖 질문의 답이 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」인 것을 확인합니다.
    - 검색은 상위 2개 청크와 거리 점수, 임계값은 1.5입니다.
    - 서비스 주소는 `http://127.0.0.1:8031`(포트 8031)입니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex04_s5_diagram.svg)

## 4. 단계별 요구사항

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 주어진 것

라이브러리 불러오기, `.env` 읽기, 모델 준비는 주어진 것입니다. 아래 셀을 고치지 않고 그대로 실행합니다.

- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.

In [ ]:
import csv
import json
import os
import subprocess
import sys
import time

import httpx
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from pydantic import BaseModel

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

# 주어진 자료
NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."
SYSTEM = ("너는 시설 안내 담당자다. 인사말처럼 검색이 필요 없는 말에는 바로 답한다. "
          "그 밖의 모든 질문은 반드시 faq_search 도구로 근거를 먼저 찾고, 도구 결과에 있는 내용으로만 답한다. "
          "도구 결과가 '검색 결과 없음'이면 네가 아는 지식으로 답하지 말고 "
          f"'{NO_EVIDENCE}'라고만 답한다.")

In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 검색 도구를 시험하면 통과한 청크의 행 번호와 카테고리가 모입니다.
2. 인사말의 출처 목록이 비어 있고, 환불 질문의 출처 목록에 티켓 카테고리의 행이 들어 있으며, 문서 밖 질문은 답이 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」이고 출처 목록이 비어 있습니다.
3. 서비스 응답 JSON에 답과 출처 목록 두 칸이 있고 값이 노트북 실행과 같은 성격입니다.

세 가지가 모두 확인되면 완성입니다.